In [ ]:
import os
from datetime import datetime
import pandas as pd
from pathlib import Path
import geopandas as gpd
from shapely.ops import nearest_points


from hydromt_sfincs import SfincsModel, DATADIR
from hydromt._utils import log

import cartopy

In [ ]:
my_catalog = "../data/catalog.yml"

In [ ]:
sf = SfincsModel(
    data_libs=["deltares_data", my_catalog],  # specify which data libraries to use
    root="../models/ahr_river_v03_Marg_additionalsrc_100m_cutpolygon",  # dedicated root for 100m cut-polygon model
    mode="w+",  # specify the mode for opening the model (r=read only, r+=append, w=write, w+=overwrite
    write_gis=True,  # specify whether to write GIS data
)

In [ ]:
sf.grid.create_from_region(
    region={"geom": "../data/catchment/ahr_catchment_Marg_additionalsrc_cutpolygon.geojson"},  # specify the region geometry
    res=100,  # specify the resolution
    rotated=False,  # specify whether the grid is rotated try without roation
    crs="utm",  # specify the coordinate reference system, when "utm", the UTM zone is automatically detected
)

In [ ]:
elevation_list = [{"elevation": "dtm_1m"}]

# Add depth information to modelgrid based on these chosen datasets
sf.elevation.create(elevation_list=elevation_list)

In [ ]:
sf.mask.create_active(
    include_polygon="../data/catchment/ahr_catchment_Marg_additionalsrc_cutpolygon.geojson",
)

In [ ]:

_ = sf.plot_basemap(
    variable="mask", plot_region=False, plot_bounds=True, bmap="sat", zoomlevel=12
)


In [ ]:
# Here we add outflow cells, only where clicked in shapefile along part of the lateral boundaries
gdf_include = sf.data_catalog.get_geodataframe(
    "../data/catchment/outflow_mask.shp"
)

sf.mask.create_boundary(
    btype="outflow",
    include_polygon=gdf_include,
    reset_bounds=True,
)

**Inflow: from the discharge observation points, added a part that makes sure they are on the polygon boundary avoiding the risk of being discarted if outide of the boundary**

In [ ]:
# Add inflow using the same source points/timeseries used by prepare_src_points and update_ahr_river_initial_cond


project_root = Path.cwd().parent  # repository root (run notebooks from notebooks/)

dis_ts = pd.read_csv(
    project_root / "data" / "observations" / "src_points_ts_Marg_additionalsrc.csv",
    index_col=0,
    parse_dates=[0],
)

src_locs = gpd.read_file(project_root / "data" / "observations" / "src_points_locs_Marg_additionalsrc_reprojected.geojson")
src_locs = src_locs.set_index("id")

# Keep source IDs as integer index; do not convert all original timeseries columns upfront
src_locs.index = pd.to_numeric(src_locs.index, errors="coerce").astype("Int64")
if src_locs.index.isna().any():
    raise ValueError("Found non-numeric IDs in src_points_locs_Marg_additionalsrc_reprojected.geojson")
src_locs.index = src_locs.index.astype(int)

# Map integer source IDs to the original timeseries column labels
orig_cols = pd.Index(dis_ts.columns)
dis_col_ids = pd.to_numeric(orig_cols, errors="coerce")
valid_col_mask = ~dis_col_ids.isna()

id_to_col = {
    int(col_id): col_name
    for col_id, col_name, is_valid in zip(dis_col_ids, orig_cols, valid_col_mask)
    if is_valid
}

# Keep only IDs that exist in both files and preserve source-location order
common_ids = [src_id for src_id in src_locs.index if src_id in id_to_col]
if len(common_ids) == 0:
    raise ValueError("No common source IDs between src_points_ts_Marg_additionalsrc.csv and src_points_locs_Marg_additionalsrc_reprojected.geojson")

matched_cols = [id_to_col[src_id] for src_id in common_ids]
matched_col_set = set(matched_cols)
non_numeric_cols = [c for c, ok in zip(orig_cols, valid_col_mask) if not ok]
unmatched_numeric_cols = [
    c for c, ok in zip(orig_cols, valid_col_mask)
    if ok and c not in matched_col_set
]
all_dropped_cols = [c for c in orig_cols if c not in matched_col_set]

dis_ts = dis_ts.loc[:, matched_cols]
dis_ts.columns = common_ids  # selected forcing columns now use integer IDs
src_locs = src_locs.loc[common_ids].copy()

print(f"Kept columns (matched to source IDs): {len(matched_cols)}")
print(f"Dropped columns total: {len(all_dropped_cols)}")
print(f"Dropped non-numeric columns: {non_numeric_cols if non_numeric_cols else 'None'}")
print(f"Dropped numeric-but-unmatched columns: {unmatched_numeric_cols if unmatched_numeric_cols else 'None'}")

# Snap points to the cut-polygon boundary when they are close enough
cut_poly = gpd.read_file(project_root / "data" / "catchment" / "ahr_catchment_Marg_additionalsrc_cutpolygon.geojson")
if src_locs.crs != cut_poly.crs:
    src_locs = src_locs.to_crs(cut_poly.crs)

boundary_geom = cut_poly.unary_union.boundary
snap_tolerance_m = 50.0  # for 100 m grid, ~0.5 cell size

dist_to_boundary = src_locs.geometry.distance(boundary_geom)
snapped_geom = []
snapped_mask = []

for geom, dist in zip(src_locs.geometry, dist_to_boundary):
    if dist <= snap_tolerance_m:
        snapped_geom.append(nearest_points(geom, boundary_geom)[1])
        snapped_mask.append(True)
    else:
        snapped_geom.append(geom)
        snapped_mask.append(False)

src_locs["dist_to_boundary_m"] = dist_to_boundary
src_locs["snapped_to_boundary"] = snapped_mask
src_locs.geometry = snapped_geom

print(f"Source points: {len(src_locs)}")
print(f"Snapped to boundary (<= {snap_tolerance_m} m): {int(src_locs['snapped_to_boundary'].sum())}")
print(f"Not snapped: {int((~src_locs['snapped_to_boundary']).sum())}")

# Final safety alignment before assigning forcing
common_ids = [c for c in dis_ts.columns if c in src_locs.index]
dis_ts = dis_ts.loc[:, common_ids]
src_locs = src_locs.loc[common_ids].copy()
dis_ts = dis_ts.sort_index()

In [ ]:
# Ensure timeseries overlaps model time window to avoid empty data after slicing

sf_config = getattr(sf, "config", None)
if sf_config is None:
    raise AttributeError("Model has no 'config' object.")

# Fixed model window requested by user
model_start = pd.to_datetime("2021-07-12 00:00:00")
model_end = pd.to_datetime("2021-07-17 00:00:00")

# Update the actual SFINCS config store used by downstream HydroMT components
sf.config.update(
    {
        "tref": model_start,
        "tstart": model_start,
        "tstop": model_end,
        "trstout": 172800,
    }
)

# Read timing back from config after update
tstart = pd.to_datetime(sf.config.get("tstart"))
tstop = pd.to_datetime(sf.config.get("tstop"))

# Keep only data inside model time window
dis_ts_in_window = dis_ts.loc[(dis_ts.index >= tstart) & (dis_ts.index <= tstop)]

if dis_ts_in_window.empty:
    # Build model-period index and fill values from nearest available observations
    if len(dis_ts.index) >= 2:
        dt = dis_ts.index.to_series().diff().dropna().median()
        if pd.isna(dt) or dt <= pd.Timedelta(0):
            dt = pd.Timedelta(hours=1)
    else:
        dt = pd.Timedelta(hours=1)

    model_index = pd.date_range(tstart, tstop, freq=dt)
    if len(model_index) == 0:
        model_index = pd.DatetimeIndex([tstart, tstop]).unique().sort_values()

    dis_ts = (
        dis_ts.reindex(dis_ts.index.union(model_index))
        .sort_index()
        .interpolate(method="time", limit_direction="both")
        .ffill()
        .bfill()
        .reindex(model_index)
    )
    print(f"Dis_ts: {dis_ts.shape[0]} values")
    print(
        f"Timeseries had no overlap with model window [{tstart} - {tstop}]. "
        "Generated a filled timeseries on model period."
    )
else:
    dis_ts = dis_ts_in_window

if dis_ts.empty:
    raise ValueError("Discharge timeseries is empty after time alignment.")

# Set discharge points (API differs between hydromt-sfincs versions)
dp = sf.discharge_points
set_ok = False

print(tstart, tstop)
print(dis_ts.head())
print(dis_ts.tail())

In [ ]:
# Fix time index name (HydroMT expects "time")
dis_ts.index.name = "time"

# Ensure IDs match between locations and timeseries
dis_ts.columns = dis_ts.columns.astype(int)
src_locs.index = src_locs.index.astype(int)

print("dis_ts columns:", dis_ts.columns.tolist())
print("src_locs index:", src_locs.index.tolist())
print("Missing in dis_ts:", set(src_locs.index) - set(dis_ts.columns))
print("Extra in dis_ts:", set(dis_ts.columns) - set(src_locs.index))

# Align order of columns with location IDs
dis_ts = dis_ts[src_locs.index]

# --- Set discharge points ---
dp.set(dis_ts, src_locs)

print("Discharge points successfully set.")

# # Ensure dis_ts columns are labeled with source IDs, not positions
# dis_ts.columns = src_locs.index.tolist()
# print("Corrected dis_ts columns to source IDs:", dis_ts.columns.tolist())

In [ ]:
# Debug mapping between internal order and original source IDs

missing = [name for name in ["sf", "src_locs", "dis_ts"] if name not in globals()]
if missing:
    print("Missing variables in current kernel:", missing)
    print("Run Cell 10 (prepare src/discharge), Cell 11 (time alignment), and Cell 12 (set discharge points), then rerun this cell.")
else:
    # Original IDs in the order passed to discharge_points
    orig_ids = pd.Index(src_locs.index).astype(int).tolist()
    map_df = pd.DataFrame(
        {
            "internal_index": range(len(orig_ids)),
            "original_source_id": orig_ids,
        }
    )

    print("dis_ts columns (input IDs):", pd.Index(dis_ts.columns).astype(int).tolist())
    print("src_locs index (input IDs):", orig_ids)
    print("\nInternal -> original ID mapping used when passed to HydroMT:")
    print(map_df.to_string(index=False))

    # Optional: inspect what HydroMT stored internally, if available
    locs = getattr(sf.discharge_points, "locations", None)
    if locs is None:
        print("\nNo sf.discharge_points.locations attribute found in this version.")
    else:
        print("\nsf.discharge_points.locations preview:")
        cols = [c for c in ["id", "name", "station_id"] if c in locs.columns]
        if cols:
            print(locs[cols].head())
        else:
            print(locs.head())

In [ ]:
dis_ts.index

In [ ]:
roughness_list = [{"lulc": "esa_worldcover"}]  # ESA WorldCover with HydroMT built-in mapping and VRT

sf.subgrid.create(
    elevation_list=elevation_list,
    roughness_list=roughness_list,
    nr_subgrid_pixels=10,
    nrmax=4000,
    write_dep_tif=True,
    write_man_tif=True,
    buffer_cells=0, 
    rgh_lev_land = 0.0, 
    manning_land=0.04,
)

In [ ]:
sf.plot_basemap(
    variable="dep", plot_geoms=True, plot_bounds=True, bmap="sat", zoomlevel=12
)

In [ ]:
# Add observation points for sfincs_his.nc output before writing model files

if "sf" not in globals():
    raise NameError("Missing required variable 'sf'. Run previous setup cells first.")

project_root = Path.cwd().parent  # repository root (run notebooks from notebooks/)
gauges_locs = project_root / "data" / "observations" / "gauges_locs.geojson"
if not gauges_locs.exists():
    raise FileNotFoundError(f"Gauge locations file not found: {gauges_locs}")

obs_ok = False

# Avoid hasattr(sf, ...) because HydroMT Model.__getattr__ may raise KeyError
setup_obs = getattr(type(sf), "setup_observation_points", None)
if callable(setup_obs):
    sf.setup_observation_points(locations=gauges_locs, merge=False)
    obs_ok = True
    print("Observation points configured using sf.setup_observation_points()")
else:
    obs_comp = None
    if hasattr(sf, "get_component"):
        try:
            obs_comp = sf.get_component("observation_points")
            print("Found observation_points component via get_component()")
        except Exception:
            obs_comp = None

    if obs_comp is not None:
        create_fn = getattr(obs_comp, "create", None)
        if callable(create_fn):
            create_fn(locations=gauges_locs, merge=False)
            obs_ok = True
            

if not obs_ok:
    raise AttributeError("Could not set observation points with available API on this hydromt-sfincs version.")

print("Observation points configured from:", gauges_locs)
print("Discharge forcing remains as configured in previous cells.")
print("Now run the next cell: sf.write()")

In [ ]:
sf.write()